# Project: A Multi-Source Knowledge Base with Routing

This project builds a router that classifies an incoming query, dispatches it to the relevant
specialized agents **in parallel**, and synthesizes their answers into one response. It
coordinates three specialists:

- A **GitHub agent** that searches code, issues, and pull requests.
- A **Notion agent** that searches internal documentation and wikis.
- A **Slack agent** that searches relevant threads and discussions.

When a user asks "How do I authenticate API requests?", the router decomposes the query into
source-specific sub-questions, routes them to the relevant agents at the same time, and
combines the results into a coherent answer.

```mermaid
graph LR
    A([Query]) --> B[Classify]
    B --> C[GitHub agent]
    B --> D[Notion agent]
    B --> E[Slack agent]
    C --> F[Synthesize]
    D --> F
    E --> F
    F --> G([Combined answer])

    classDef trigger fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:2px,color:#030710
    class A,G trigger
    class B,C,D,E,F process
```

### Router vs. Subagents

The Subagents pattern (previous project) can also route to multiple agents, but the supervisor
*itself* decides, as a live agent, which to call and when. The router pattern instead uses a
**dedicated classification step** -- often one structured-output call, not a full agent -- that
decides once, dispatches in parallel, and hands off to a separate synthesis step. Use a router
when you want explicit control over parallel execution and lightweight, rule-based or
structured classification; use Subagents when you want the model to decide dynamically, turn
by turn.

## Setup

Copy `.env.example` to `.env` and set `OPENAI_API_KEY=sk-...` before running this notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Copy .env.example to .env and add your key.")

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
router_llm = init_chat_model("openai:gpt-5-mini")
print("Models ready.")


## 1. Define state

Three state shapes, each with a distinct job:

- `AgentInput`: the simple input each specialist agent receives (just a query).
- `AgentOutput`: the output each specialist returns (source name + result).
- `RouterState`: the overall workflow state.

```mermaid
classDiagram
    class AgentInput {
        query: str
    }
    class AgentOutput {
        source: str
        result: str
    }
    class RouterState {
        query: str
        classifications: list
        results: list (reducer: operator.add)
        final_answer: str
    }
```


In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
import operator

class AgentInput(TypedDict):
    """Simple input state for each specialist agent."""
    query: str

class AgentOutput(TypedDict):
    """Output from each specialist agent."""
    source: str
    result: str

class Classification(TypedDict):
    """A single routing decision: which agent to call with what query."""
    source: Literal["github", "notion", "slack"]
    query: str

class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]  # reducer collects parallel results
    final_answer: str

print("State schemas defined.")


The `results` field uses a **reducer** (`operator.add`) to merge outputs from multiple
parallel agent executions into a single list. Without a reducer here, LangGraph would raise an
error the moment two parallel branches both tried to write to the same key in one step.

## 2. Define tools for each vertical

Stub implementations stand in for real API calls (GitHub's API, Notion's API, Slack's API),
so the whole notebook runs without any of those services configured.


In [ ]:
from langchain.tools import tool

@tool
def search_code(query: str, repo: str = "main") -> str:
    """Search code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"

@tool
def search_issues(query: str) -> str:
    """Search GitHub issues and pull requests."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"

@tool
def search_prs(query: str) -> str:
    """Search pull requests for implementation details."""
    return "PR #156 added JWT authentication, PR #178 updated OAuth scopes"

@tool
def search_notion(query: str) -> str:
    """Search Notion workspace for documentation."""
    return "Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"

@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return "Page content: Step-by-step authentication setup instructions"

@tool
def search_slack(query: str) -> str:
    """Search Slack messages and threads."""
    return "Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"

@tool
def get_thread(thread_id: str) -> str:
    """Get a specific Slack thread."""
    return "Thread discusses best practices for API key rotation"

print("Seven tools defined across three verticals.")


## 3. Create a specialized agent per vertical

All three follow the same shape -- only the tools and system prompt differ.


In [ ]:
from langchain.agents import create_agent

github_agent = create_agent(
    model,
    tools=[search_code, search_issues, search_prs],
    system_prompt=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    ),
)

notion_agent = create_agent(
    model,
    tools=[search_notion, get_page],
    system_prompt=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    ),
)

slack_agent = create_agent(
    model,
    tools=[search_slack, get_thread],
    system_prompt=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    ),
)
print("Three specialist agents created.")


## 4. Build the router workflow

Four steps: **classify** (structured output decides which sources are relevant and generates a
targeted sub-question for each), **route** (fan out via `Send`), **query** (each agent answers
its sub-question), **synthesize** (combine into one response).

```mermaid
graph TD
    A[classify_query] -->|Send per classification| B1[query_github]
    A -->|Send| B2[query_notion]
    A -->|Send| B3[query_slack]
    B1 --> C[synthesize_results]
    B2 --> C
    B3 --> C
```

> **A real gotcha worth knowing:** `ClassificationResult` below wraps `Classification` (a
> `TypedDict`) inside a Pydantic model so `with_structured_output` can enforce it. On Python
> versions before 3.12, Pydantic v2 refuses a `TypedDict` imported from the standard `typing`
> module in that position -- it raises *"Please use `typing_extensions.TypedDict` instead of
> `typing.TypedDict` on Python < 3.12."* That's why the state-schema cell above imports
> `TypedDict` from `typing_extensions` rather than `typing`, even though plain `TypedDict`
> classes that never reach Pydantic would have worked fine with either.


In [ ]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ClassificationResult(BaseModel):
    """Result of classifying a user query into agent-specific sub-questions."""
    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )

def classify_query(state: RouterState) -> dict:
    """Classify query and determine which agents to invoke."""
    structured_llm = router_llm.with_structured_output(ClassificationResult)
    result = structured_llm.invoke([
        {
            "role": "system",
            "content": """Analyze this query and determine which knowledge bases to consult.
For each relevant source, generate a targeted sub-question optimized for that source.

Available sources:
- github: Code, API references, implementation details, issues, pull requests
- notion: Internal documentation, processes, policies, team wikis
- slack: Team discussions, informal knowledge sharing, recent conversations

Return ONLY the sources that are relevant to the query. Each source should have
a targeted sub-question optimized for that specific knowledge domain.""",
        },
        {"role": "user", "content": state["query"]},
    ])
    return {"classifications": result.classifications}


def route_to_agents(state: RouterState) -> list[Send]:
    """Fan out to agents based on classifications."""
    return [Send(c["source"], {"query": c["query"]}) for c in state["classifications"]]


def query_github(state: AgentInput) -> dict:
    """Query the GitHub agent."""
    result = github_agent.invoke({"messages": [{"role": "user", "content": state["query"]}]})
    return {"results": [{"source": "github", "result": result["messages"][-1].content}]}


def query_notion(state: AgentInput) -> dict:
    """Query the Notion agent."""
    result = notion_agent.invoke({"messages": [{"role": "user", "content": state["query"]}]})
    return {"results": [{"source": "notion", "result": result["messages"][-1].content}]}


def query_slack(state: AgentInput) -> dict:
    """Query the Slack agent."""
    result = slack_agent.invoke({"messages": [{"role": "user", "content": state["query"]}]})
    return {"results": [{"source": "slack", "result": result["messages"][-1].content}]}


def synthesize_results(state: RouterState) -> dict:
    """Combine results from all agents into a coherent answer."""
    if not state["results"]:
        return {"final_answer": "No results found from any knowledge source."}
    formatted = [f"**From {r['source'].title()}:**\n{r['result']}" for r in state["results"]]
    synthesis_response = router_llm.invoke([
        {
            "role": "system",
            "content": f"""Synthesize these search results to answer the original question: "{state['query']}"

- Combine information from multiple sources without redundancy
- Highlight the most relevant and actionable information
- Note any discrepancies between sources
- Keep the response concise and well-organized""",
        },
        {"role": "user", "content": "\n\n".join(formatted)},
    ])
    return {"final_answer": synthesis_response.content}

print("Workflow nodes defined.")


## 5. Compile the workflow

`add_conditional_edges` with `route_to_agents` is what enables parallel execution: when that
function returns multiple `Send` objects, LangGraph runs all of those nodes at once.


In [ ]:
workflow = (
    StateGraph(RouterState)
    .add_node("classify", classify_query)
    .add_node("github", query_github)
    .add_node("notion", query_notion)
    .add_node("slack", query_slack)
    .add_node("synthesize", synthesize_results)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route_to_agents, ["github", "notion", "slack"])
    .add_edge("github", "synthesize")
    .add_edge("notion", "synthesize")
    .add_edge("slack", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)
print("Workflow compiled.")


## 6. Use the router


In [ ]:
result = workflow.invoke({"query": "How do I authenticate API requests?"})

print("Original query:", result["query"])
print("\nClassifications:")
for c in result["classifications"]:
    print(f"  {c['source']}: {c['query']}")
print("\n" + "=" * 60 + "\n")
print("Final Answer:")
print(result["final_answer"])


The router analyzed the query, classified it (probably GitHub and Notion for this technical
question, skipping Slack), queried the relevant agents in parallel, and synthesized the
results into one coherent answer.

## 7. Understanding the architecture

### Classification phase

`classify_query` uses **structured output** (a Pydantic model, enforced by
`with_structured_output`) rather than free-form text parsing -- this is more reliable and makes
the routing logic explicit. Irrelevant sources are simply omitted from the result.

### Parallel execution with `Send`

```python
# classifications: [{"source": "github", "query": "..."}, {"source": "notion", "query": "..."}]
# becomes:
[Send("github", {"query": "..."}), Send("notion", {"query": "..."})]
# Both agents execute at the same time, each receiving only the query it needs.
```

Each agent node receives a plain `AgentInput` with just a `query` field -- not the full router
state. That keeps the interface between the router and each specialist clean and explicit.

### Result collection with reducers

Each agent returns `{"results": [{"source": ..., "result": ...}]}`, and the `operator.add`
reducer concatenates these lists as they arrive -- however many branches ran in parallel.

### Synthesis phase

LangGraph automatically waits for every dispatched branch to finish before running
`synthesize`, which then references the original query to make sure the final answer actually
addresses what was asked.

## 8. Advanced: stateful routers

The router above is **stateless** -- every call is independent, with no memory between turns.
For a multi-turn conversation, the simplest fix is to wrap the stateless router as a tool that
a conversational agent calls. The agent handles memory; the router stays a clean, reusable step.

```mermaid
graph LR
    U([Multi-turn conversation]) --> CA[Conversational agent<br/>with checkpointer]
    CA -->|search_knowledge_base tool| W[Stateless router workflow]
    W -.-> CA
```


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_knowledge_base(query: str) -> str:
    """Search across multiple knowledge sources (GitHub, Notion, Slack).

    Use this to find information about code, documentation, or team discussions.
    """
    result = workflow.invoke({"query": query})
    return result["final_answer"]

conversational_agent = create_agent(
    model,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a helpful assistant that answers questions about our organization. "
        "Use the search_knowledge_base tool to find information across our code, "
        "documentation, and team discussions."
    ),
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "user-123"}}
result = conversational_agent.invoke(
    {"messages": [{"role": "user", "content": "How do I authenticate API requests?"}]}, config
)
print(result["messages"][-1].content)


In [ ]:
result = conversational_agent.invoke(
    {"messages": [{"role": "user", "content": "What about rate limiting for those endpoints?"}]}, config
)
print(result["messages"][-1].content)


The tool-wrapper approach is recommended for most cases: it keeps a clean separation between
the router (handles multi-source querying) and the conversational agent (handles context and
memory). A full-persistence approach -- where the router itself holds message history -- adds
real complexity, and usually the Handoffs or Subagents patterns give clearer multi-turn
semantics instead.

## Key takeaways

| Point | Why it matters |
|---|---|
| Classification is a structured-output call, not a full agent | Cheaper, more reliable, and explicit about routing logic |
| `Send(node, payload)` fans out in parallel | All matched sources are queried at the same time, not one after another |
| State keys written by parallel branches need a reducer | `Annotated[list[...], operator.add]` -- otherwise LangGraph raises an error |
| Each agent node gets a narrow `AgentInput`, not full state | Keeps specialist agents decoupled from router internals |
| Wrap a stateless router as a tool for multi-turn use | Simplest way to add memory without rewriting the router itself |

**When to use this pattern:** independent knowledge sources or verticals that benefit from
parallel querying, with results combined into one response. For a workflow where the *same*
agent's behavior should change as a task progresses, see the Handoffs notebook instead.
